# Feature hashing: the hashing trick for huge category spaces

_Feature Engineering_

**Hash millions of categories into a fixed number of buckets — no vocabulary stored, no matter how many categories appear.**

Real data is full of categorical features with a gigantic number of possible values: every
       distinct word in a corpus, every user ID, every advertiser ID, every IP address. The usual way to feed
       a category to a model is one-hot encoding &mdash; one column per distinct value, with a single 1
       marking which value occurred. That is fine for a handful of categories. It is hopeless when there are
       millions: the feature vector becomes millions of columns wide, and you must store a giant
       vocabulary (a lookup table mapping each value to its column).

       Feature hashing &mdash; the "hashing trick" &mdash; sidesteps both problems. Fix a
       number of output buckets $m$ ahead of time (say a million). For each category, run it through a
       hash function $h$ that returns a bucket index between $0$ and $m-1$, and add $1$ to that bucket.
       The output is always an $m$-dimensional vector, no matter how many distinct categories exist,
       and you never store a vocabulary &mdash; the hash function is the mapping. This is Chapter 5 of
       Zheng & Casari's Feature Engineering for Machine Learning, and their running example is the
       Yelp reviews text.

---

This notebook builds the hashing trick up one step at a time. Run each cell, read the note above it, and watch the feature matrix shrink from a 50,000-column vocabulary to a fixed handful of buckets — then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

### Step 1 — Build a high-cardinality categorical dataset

We synthesize a realistic feature with a *huge* number of possible values: 50,000 distinct user-id strings. Only a few thousand of them ever actually fire, but one-hot encoding still has to reserve a column for every id it might ever see. Each id secretly belongs to one of two groups, and that group decides the label (with a little noise) — so the category really does carry learnable signal.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction import FeatureHasher
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(0)   # seeded so the numbers reproduce

N_IDS = 50_000          # size of the POSSIBLE id space (what one-hot must reserve)
N_ROWS = 200_000        # number of training/eval events

# Hidden group (0/1) for every possible id -> this is the signal.
id_group = rng.integers(0, 2, size=N_IDS)

# Only 4,000 ids actually fire, each repeating across many events, so the
# per-id signal is genuinely learnable. They are still drawn from the full
# 50,000-wide id space that one-hot would have to reserve columns for.
active = rng.choice(N_IDS, size=4_000, replace=False)
row_ids = rng.choice(active, size=N_ROWS, replace=True)   # which id fired per event

# Label = the id's hidden group, with 5% label noise so it is not trivial.
y = id_group[row_ids].copy()
flip = rng.random(N_ROWS) < 0.05
y[flip] = 1 - y[flip]

# The raw categorical strings the model sees, e.g. "user_03917".
cats = np.array([f"user_{i:05d}" for i in row_ids])

Xtr_cat, Xte_cat, ytr, yte = train_test_split(
    cats, y, test_size=0.25, random_state=0, stratify=y)

n_distinct = len(np.unique(cats))
print(f"STEP 1  {N_ROWS} events, {n_distinct} distinct ids seen "
      f"(out of {N_IDS} possible). High-cardinality categorical.")

### Step 2 — Reproduce the one-hot problem

Out of 50,000 possible ids only a few thousand ever appear, yet one-hot encoding reserves a column for *all* of them — a giant, almost-empty matrix plus a stored vocabulary of the same size. We chart how many id columns sit empty, then one-hot encode against the full id space, measure the width and the memory one dense row would take, and train a linear model on it.

In [ ]:
# Left plot: how many of the 50,000 possible id columns ever fire vs stay empty.
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
counts = np.bincount(row_ids, minlength=N_IDS)     # times each id fired
n_seen = int((counts > 0).sum())
n_unseen = N_IDS - n_seen
ax[0].bar(["appeared", "never seen\n(still reserved)"], [n_seen, n_unseen],
          color=["#7f8c8d", "#bdc3c7"])
ax[0].set_title("STEP 2: most of the 50,000 id columns are empty\n"
                "(one-hot reserves a column for every possible id)")
ax[0].set_ylabel("number of ids")
for i, v in enumerate([n_seen, n_unseen]):
    ax[0].text(i, v, f"{v}", ha="center", va="bottom")

# One-hot encode against the FULL possible id space (production reality: you
# must reserve a column for every id you might ever see).
all_ids = np.array([f"user_{i:05d}" for i in range(N_IDS)])   # the full vocabulary
ohe = OneHotEncoder(categories=[all_ids], handle_unknown="ignore")
Xtr_oh = ohe.fit_transform(Xtr_cat.reshape(-1, 1))   # sparse, but width is huge
Xte_oh = ohe.transform(Xte_cat.reshape(-1, 1))

oh_width = Xtr_oh.shape[1]
oh_vocab = oh_width                       # one stored vocabulary entry per column
oh_dense_row_kb = oh_width * 8 / 1024     # bytes if ONE row were dense (float64)

clf_oh = LogisticRegression(max_iter=1000)
clf_oh.fit(Xtr_oh, ytr)
acc_oh = clf_oh.score(Xte_oh, yte)

print(f"STEP 2 PROBLEM  one-hot width = {oh_width} columns, "
      f"stored vocabulary = {oh_vocab} entries.")
print(f"                a single dense row would need {oh_dense_row_kb:,.1f} KB "
      f"(mostly zeros). Width grows with cardinality.")
print(f"                one-hot accuracy = {acc_oh:.3f}")

### Step 3 — Apply feature hashing

`FeatureHasher` maps each string to `hash(string) mod K` for a **fixed** number of buckets `K`. No matter how many distinct ids exist, the width stays `K` and no vocabulary is stored. We hash the categories, then chart how the distinct ids spread over the buckets: most buckets hold 0 or 1 id, so collisions (two ids sharing a bucket) are rare.

In [ ]:
K = 2 ** 14   # 16,384 buckets — a fixed width, still far below the 50,000 id space
hasher = FeatureHasher(n_features=K, input_type="string")

# FeatureHasher wants an iterable of token lists per row; here one token (the id) each.
Xtr_h = hasher.transform([[c] for c in Xtr_cat])
Xte_h = hasher.transform([[c] for c in Xte_cat])
h_width = Xtr_h.shape[1]

# Right plot: how the distinct ids spread over the K buckets.
seen_ids = np.unique(row_ids)
bucket_of_id = hasher.transform([[f"user_{i:05d}"] for i in seen_ids])
ids_per_bucket = np.asarray(np.abs(bucket_of_id).sum(axis=0)).ravel()  # ids in each bucket
load_hist = np.bincount(ids_per_bucket.astype(int))   # #buckets holding 0,1,2,... ids
ax[1].bar(range(len(load_hist)), load_hist, color="#2980b9")
ax[1].set_title(f"STEP 3: {len(seen_ids)} ids hashed into a FIXED {K} buckets\n"
                f"(x = ids sharing a bucket, y = #buckets — collisions are rare)")
ax[1].set_xlabel("ids landing in the same bucket")
ax[1].set_ylabel("number of buckets")
plt.tight_layout()
plt.show()

### Step 4 — Show the fix: same accuracy, fixed width, no vocabulary

We train the *same* linear model on the fixed-width hashed features. The width has collapsed from 50,000 to 16,384, no vocabulary is stored, and accuracy stays about the same — the occasional collision costs little. The two side-by-side bar charts make the trade explicit: feature-matrix width (one-hot grows with cardinality, hashed is fixed) and held-out accuracy (nearly identical).

In [ ]:
clf_h = LogisticRegression(max_iter=1000)
clf_h.fit(Xtr_h, ytr)
acc_h = clf_h.score(Xte_h, yte)

print(f"STEP 4 FIX      hashed width = {h_width} columns (FIXED), "
      f"vocabulary stored = 0 entries.")
print(f"                hashed accuracy = {acc_h:.3f}")
print(f"\nWidth: {oh_width} cols (one-hot)  ->  {h_width} cols (hashed)   "
      f"= {oh_width / h_width:.1f}x narrower, no vocabulary.")
print(f"PROBLEM (one-hot acc): {acc_oh:.3f}   ->   FIX (hashed acc): {acc_h:.3f}")

# Side-by-side comparison: WIDTH (left) and ACCURACY (right).
fig2, ax2 = plt.subplots(1, 2, figsize=(11, 4.4))
ax2[0].bar(["one-hot", "hashed"], [oh_width, h_width], color=["#c0392b", "#2980b9"])
ax2[0].set_title("Feature-matrix WIDTH (columns)\nlower is cheaper")
ax2[0].set_ylabel("number of columns")
for i, v in enumerate([oh_width, h_width]):
    ax2[0].text(i, v, f"{v}", ha="center", va="bottom")
ax2[1].bar(["one-hot", "hashed"], [acc_oh, acc_h], color=["#c0392b", "#2980b9"])
ax2[1].set_title("Held-out ACCURACY\nabout the same")
ax2[1].set_ylim(0, 1)
for i, v in enumerate([acc_oh, acc_h]):
    ax2[1].text(i, v, f"{v:.3f}", ha="center", va="bottom")
plt.tight_layout()
plt.show()

## Reference implementation — scikit-learn

### Step 1 — Load the Yelp reviews and hash their words

The book's running example hashes the words of Yelp reviews. We load the reviews, treat each review's lowercased word list as its bag of category tokens, and run them through a `FeatureHasher` with a fixed `2**18 = 262,144` buckets. The output shape is fixed at that width no matter how large the underlying vocabulary is, and no vocabulary is ever built.

In [ ]:
import pandas as pd
import json
from sklearn.feature_extraction import FeatureHasher

# --- Load the Yelp reviews (Yelp Dataset Challenge, round 6) ---
# Get it from the book's repo: github.com/alicezheng/feature-engineering-book
f = open('yelp_academic_dataset_review.json')
review_df = pd.DataFrame([json.loads(x) for x in f.readlines()])
f.close()

# Each review's "category" tokens are simply its words.
# (For a true bag-of-words you'd tokenize; the book hashes the word list directly.)
def word_list(text):
    return text.lower().split()

categories = (word_list(t) for t in review_df['text'])

# === Feature hashing ===
# Fixed output size m = 2**18 buckets, NO vocabulary is built or stored.
h = FeatureHasher(n_features=2 ** 18, input_type='string')
f = h.transform(categories)        # sparse (n_reviews x 262144) matrix

print('hashed shape :', f.shape)   # -> (n_reviews, 262144), independent of vocab size

### Step 2 — Measure how little the hashed matrix stores

The hashed matrix is sparse, so it only stores its non-zero entries. We count those non-zeros and the bytes they occupy. Compare that to a full bag-of-words matrix, which needs a column per distinct word *and* the stored vocabulary of tens of thousands of terms — the hashed version is a fixed `2**18` wide with the vocabulary thrown away.

In [ ]:
# The hashed matrix is sparse; measure how many bytes it actually occupies.
from sys import getsizeof

print('Our pandas DataFrame is', getsizeof(review_df), 'bytes')
print('hashed feature matrix nnz :', f.nnz)                 # non-zero entries
print('hashed feature matrix size:', f.data.nbytes, 'bytes')
# A full one-hot / bag-of-words matrix needs a column per distinct word
# (a vocabulary of tens of thousands of terms) AND that stored vocabulary;
# the hashed matrix is a fixed 2**18 wide with the vocabulary thrown away.

## Visualize the data & results

_Hash a list of many category strings into m buckets for m = 16, 64, 256. As the number of buckets m grows, how does the collision rate fall — the size-vs-accuracy trade-off?_

### Step 1 — Define a collision-rate measurement

To see the size-vs-accuracy trade-off directly, we build 200 distinct category strings and a helper that hashes them into `m` buckets. A *collision* is when two or more distinct categories land in the same bucket; the helper returns the fraction of categories whose bucket is shared. We mask Python's hash to a non-negative value so the result is deterministic.

In [ ]:
import numpy as np

# 200 distinct category strings (e.g. user IDs / words). Real, deterministic.
cats = ['cat_%04d' % i for i in range(200)]

def collision_rate(categories, m):
    # Hash each category into one of m buckets (Python hash, made deterministic).
    buckets = np.array([(hash(c) & 0x7fffffff) % m for c in categories])
    # Count distinct categories per bucket; a bucket with 2+ is a collision.
    counts = np.bincount(buckets, minlength=m)
    collided = counts[buckets] > 1           # is this category's bucket shared?
    return collided.mean()

### Step 2 — Watch the collision rate fall as buckets grow

Sweeping `m` from 16 up to 256 shows the core trade-off: more buckets mean fewer collisions, at the cost of a wider feature vector. With only 16 buckets almost every one of the 200 categories collides; by 256 buckets the rate has dropped sharply. This is exactly why you pick a large `m` in practice.

In [ ]:
for m in [16, 32, 64, 128, 256]:
    print('m=%3d  collision rate = %.2f' % (m, collision_rate(cats, m)))
# -> m= 16  collision rate = 0.92
#    m= 32  collision rate = 0.78
#    m= 64  collision rate = 0.55
#    m=128  collision rate = 0.32
#    m=256  collision rate = 0.18
# More buckets -> fewer collisions, at the cost of a wider feature vector.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You have an ad-tech model with about 50 million distinct user IDs as a categorical feature, and new IDs appear every hour. Why is one-hot encoding a non-starter, and how does feature hashing fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Count what one-hot would cost: one column per distinct ID plus a dictionary mapping each ID to its column. — _With 50 million IDs the vector is 50 million wide and the stored vocabulary is enormous — and it must grow every hour as new IDs arrive._
- Replace the lookup table with a hash function $h$ and a fixed bucket count $m$. — _Each ID's bucket is computed as $h(\text{id})\bmod m$ on demand, so there is nothing to store and the width is pinned to $m$._
- Note that a brand-new ID just hashes like any other. — _The transform is stateless, so streaming IDs need no vocabulary update — ideal for online learning._

**Answer:** One-hot needs a column per ID and a giant, ever-growing vocabulary — 50 million columns plus a dictionary that must be updated hourly. Feature hashing fixes the width at a chosen $m$ and stores no table: each ID's bucket is $h(\text{id})\bmod m$, recomputed on demand, so new streaming IDs are handled instantly. The cost is occasional collisions, kept small by picking $m$ large.

</details>

**Problem 2.** Two distinct words hash to the same bucket. With plain (unsigned) counting their contributions add up, biasing the bucket. How does the signed-hash trick reduce this bias?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write the unsigned bucket: both words add $+1$, so the bucket holds the sum of their counts. — _Their signals are merged and the bucket is systematically inflated whenever both occur._
- Attach an independent sign $\xi(c)\in\{-1,+1\}$ to each word before adding. — _Now one word may add and the other subtract, so a colliding pair can cancel instead of always piling up._
- Look at the cross term in a dot product: it is $\xi(a)\xi(b)$, equally likely $+1$ or $-1$. — _Its expected value is zero, so the hashed dot product is an unbiased estimate of the true one._

**Answer:** With the signed hash, each category carries a random sign $\xi(c)\in\{-1,+1\}$. Colliding categories then add with opposite signs about half the time, so their contributions cancel in expectation rather than accumulate. The cross term $\xi(a)\xi(b)$ has mean zero, making the hashed inner product an unbiased estimate of the true inner product despite collisions.

</details>

**Problem 3.** A teammate hashed features with $m=2^{14}$ during training but, to save space at serving time, re-hashed with $m=2^{12}$. The model's accuracy collapsed. What went wrong?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that a category's bucket is $h(c)\bmod m$ — it depends on $m$. — _Changing $m$ changes every category's bucket, so the serving features point at different columns than training did._
- Recognize the model's learned weights are tied to the training bucket assignment. — _Weight in column $j$ means nothing if column $j$ now holds different categories._
- Keep $h$ and $m$ (and any sign hash) identical at train and serve time. — _The hash function IS the feature definition; it must match everywhere for the model to work._

**Answer:** The bucket is $h(c)\bmod m$, so changing $m$ from $2^{14}$ to $2^{12}$ remapped every category to a different bucket. The model's weights were learned against the training buckets, so at serving time each weight now applies to the wrong features. The fix: use the exact same hash and $m$ (and sign hash) at train and serve time — the hash is part of the feature definition.

</details>